# Week 2 – Data Collection, Cleaning & Preprocessing
**YuvaIntern | Logistics Data Analysis**

This notebook uses the Week 1 simulated logistics dataset and demonstrates a reproducible data-quality workflow.

## 1. Data Collection
The dataset contains 1,000 simulated shipment records covering distance, weight, order volume, warehouse processing, traffic, weather, vehicle type, transportation cost, delivery time, and delivery status.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/logistics_data.csv")
print("Shape:", df.shape)
display(df.head())

## 2. Data Quality Assessment
Check missing values, duplicate records, data types, and potential outliers.

In [ ]:
print("Missing values:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df.dtypes)

## 3. Cleaning
Remove duplicates, convert dates, standardize categories, and impute missing values.

In [ ]:
df=df.drop_duplicates()
df["Shipment_Date"]=pd.to_datetime(df["Shipment_Date"],errors="coerce")
cat_cols=["Traffic_Level","Weather","Vehicle_Type","Delivery_Status"]
num_cols=["Distance_KM","Shipment_Weight_KG","Order_Volume","Warehouse_Processing_Hours","Transport_Cost","Delivery_Time_Hours"]
for col in cat_cols:
    df[col]=df[col].astype("string").str.strip().str.title()
    df[col]=df[col].fillna(df[col].mode()[0])
for col in num_cols:
    df[col]=pd.to_numeric(df[col],errors="coerce")
    df[col]=df[col].fillna(df[col].median())
print("Missing values after cleaning:")
display(df.isnull().sum())

## 4. Outlier Detection Using IQR
Potential outliers are flagged for investigation rather than automatically deleted.

In [ ]:
def iqr_outliers(data,column):
    q1=data[column].quantile(.25); q3=data[column].quantile(.75); iqr=q3-q1
    lower=q1-1.5*iqr; upper=q3+1.5*iqr
    mask=(data[column]<lower)|(data[column]>upper)
    return mask,lower,upper
for col in ["Distance_KM","Transport_Cost","Delivery_Time_Hours"]:
    mask,lower,upper=iqr_outliers(df,col)
    print(f"{col}: {mask.sum()} potential outliers | bounds=({lower:.2f}, {upper:.2f})")

## 5. Scaling Numerical Features

In [ ]:
from sklearn.preprocessing import StandardScaler
scale_cols=["Distance_KM","Shipment_Weight_KG","Order_Volume","Warehouse_Processing_Hours"]
scaler=StandardScaler()
scaled=df.copy()
scaled[scale_cols]=scaler.fit_transform(df[scale_cols])
display(scaled[scale_cols].head())

## 6. Save Cleaned Dataset

In [ ]:
df.to_csv("../data/logistics_data_cleaned.csv",index=False)
print("Cleaned dataset saved.")

## 7. Reflection
Data cleaning is essential because missing, duplicate, inconsistent, or extreme observations can distort KPIs and model performance. Outliers should be investigated before removal because they may represent genuine logistics events. A documented preprocessing pipeline makes later analysis more reliable and reproducible.